In [3]:
!pip install transformers sentence-transformers accelerate torch gradio pandas

In [4]:
import pandas as pd
import gradio as gr

from transformers import pipeline

from sentence_transformers import (
    SentenceTransformer,
    util
)

In [51]:
faq_data = {

"Question":[
"How can I pay my water bill?",
"How can I pay my electricity bill?",
"How can I pay my property tax?",
"What is NDMC 311?"
],

"Answer":[

"""Water bills can be paid through NDMC online services.
Citizens can view their bill details, make online payments,
register for e-bills, apply for reconnection or disconnection,
and access other water-related services through the NDMC portal.""",

"""Electricity bills can be paid online through NDMC services.
Citizens can view bills, apply for new connections,
request load modifications, meter testing,
and access consumer grievance services.""",

"""Property tax can be paid using NDMC online property tax services.
Citizens can view tax dues, file Property Tax Returns (PTR),
download mutation certificates, and make payments online
through the NDMC portal.""",

"""NDMC 311 is a citizen service platform that enables residents
to register complaints, report civic issues, track service requests,
and access various municipal services provided by NDMC."""
]
}

In [6]:
urls = [
    "https://www.ndmc.gov.in/",
    "https://www.ndmc.gov.in/online_service.aspx"
]

In [39]:
import requests
import re
from bs4 import BeautifulSoup

def chunk_text(text, chunk_size=500):

    return [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
    ]

knowledge = []

for url in urls:

    try:

        response = requests.get(
            url,
            timeout=10
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Remove useless page sections
        for tag in soup([
            "script",
            "style",
            "nav",
            "header",
            "footer"
        ]):
            tag.decompose()

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        text = re.sub(
            r"\s+",
            " ",
            text
        )

        # Remove common NDMC menu text
        noise_words = [
            "Home",
            "Employee Corner",
            "Like Us on Facebook",
            "Follow Us on Twitter",
            "Follow us on Instagram",
            "Subscribe on YouTube",
            "Vendor Bill Registration",
            "Download Forms",
            "View Payslip",
            "NDMC Email",
            "Online Attendance"
        ]

        for word in noise_words:
            text = text.replace(word, "")

        chunks = chunk_text(
            text,
            chunk_size=500
        )

        for chunk in chunks:

            # Skip junk chunks
            if (
                "facebook" in chunk.lower()
                or "twitter" in chunk.lower()
                or "instagram" in chunk.lower()
                or "youtube" in chunk.lower()
                or len(chunk.split()) < 30
            ):
                continue

            knowledge.append(chunk)

        print(
            f"Scraped {url}"
        )

    except Exception as e:

        print(
            f"Error: {url} -> {e}"
        )

print(
    f"\nTotal chunks collected: {len(knowledge)}"
)

Scraped https://www.ndmc.gov.in/
Scraped https://www.ndmc.gov.in/online_service.aspx

Total chunks collected: 22


In [52]:
faq_df = pd.DataFrame(faq_data)

In [45]:
knowledge = [

"""
Property Tax:
NDMC provides online property tax payment services. Citizens can view tax dues, file Property Tax Returns (PTR), pay bills online, and download mutation certificates through the NDMC portal.
""",

"""
Water Services:
NDMC provides online water bill payment facilities. Citizens can view and pay water bills, register for e-bills, apply for new connections, reconnection/disconnection, change of name, and meter testing services.
""",

"""
Electricity Services:
Citizens can pay electricity bills online, apply for new connections, request load changes, apply for meter testing, and access consumer grievance services through NDMC.
""",

"""
NDMC 311:
NDMC 311 is a citizen service platform that allows residents to register complaints, report civic issues, track requests, and access municipal services.
""",

"""
Birth and Death Certificates:
Citizens can apply for birth certificates, death certificates, still birth certificates, and child name inclusion services through NDMC.
""",

"""
Water Tanker Booking:
NDMC provides online water tanker booking services for citizens through its online services portal.
"""
]

In [42]:
all_documents = faq_df["Answer"].tolist()

all_documents.extend(
    knowledge_df["content"].tolist()
)

In [53]:
all_documents = faq_df["Answer"].tolist()

all_documents.extend(knowledge)

document_embeddings = embedding_model.encode(
    all_documents,
    convert_to_tensor=True
)

In [9]:
embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
# Rebuild documents and embeddings

all_documents = faq_df["Answer"].tolist()

all_documents.extend(
    knowledge_df["content"].tolist()
)

document_embeddings = embedding_model.encode(
    all_documents,
    convert_to_tensor=True
)

print("✅ Documents loaded:", len(all_documents))
print("✅ Embeddings created successfully")

✅ Documents loaded: 55
✅ Embeddings created successfully


In [55]:
def retrieve_answer(user_query):

    query_embedding = embedding_model.encode(
        user_query,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding,
        document_embeddings
    )[0]

    best_match = scores.argmax()

    return all_documents[
        best_match.item()
    ]

In [13]:
chatbot_model = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [49]:
def ndmc_chatbot(user_query):

    if user_query.lower() in [
        "hi",
        "hello",
        "hey"
    ]:
        return (
            "Hello! Welcome to NDMC Citizen Services. "
            "How may I assist you today?"
        )

    context = retrieve_answer(user_query)

    return f"""
Based on NDMC information:

{context}

If you need more details about this service,
please visit the official NDMC portal or contact
the concerned department.
"""

In [47]:
print(retrieve_answer("How can I pay my property tax?"))

Property Tax:
NDMC provides online property tax payment services. Citizens can view tax dues, file Property Tax Returns (PTR), pay bills online, and download mutation certificates through the NDMC portal.



Electricity Services:
Citizens can pay electricity bills online, apply for new connections, request load changes, apply for meter testing, and access consumer grievance services through NDMC.


In [37]:
interface = gr.Interface(

    fn=ndmc_chatbot,

    inputs=gr.Textbox(
        lines=4,
        placeholder="Ask NDMC related questions..."
    ),

    outputs=gr.Textbox(
        lines=15,
        label="NDMC Response"
    ),

    title="NDMC Chatbot",

    description="NDMC AI Assistant powered by RAG and TinyLlama."
)

In [56]:
interface.launch(
    share=True
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://031933f59a7ce8dfd2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
print("\nWebsite Document Example")
print("-" * 50)

print(all_documents[10][:1000])


Website Document Example
--------------------------------------------------
New Delhi Municipal Council Like Us on Facebook Follow Us on Twitter Follow us
                        on Instagram Subscribe
                        on YouTube Email Us on care@ndmc.gov.in Employee Corner Home Online Services Pay Electricity Bill Pay Water Bill New Electricity Connection New Water Connection Reconnection/Disconnection Electricity Reconnection/Disconnection Filtered Water Property Tax Estate Book Baratghar Yellow Fever Vaccination Birth Certificate Still Birth Certificate Deat
